# 🌾 Wheat Rust Disease Detection
## DeepLabV3+ with ResNet-50 Backbone — Binary Semantic Segmentation
**Dataset:** abdur548/nwrd-patched (Kaggle) | **Framework:** PyTorch + segmentation-models-pytorch

In [1]:
import os
os.environ['KAGGLE_USERNAME'] = "abdur548"
os.environ['KAGGLE_KEY']      = "a83df7326200f6f046254700891fcf1b"

In [2]:
import kagglehub


path = kagglehub.dataset_download('abdur548/nwrd-patched')
print('Dataset path:', path)

100%|██████████| 1.99G/1.99G [00:21<00:00, 98.1MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/abdur548/nwrd-patched/versions/1


In [3]:
# Cell 2 — Install dependencies
!pip install segmentation-models-pytorch albumentations wandb torchmetrics monai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 92.2 MB/s eta 0:00:00


In [4]:
# Cell 3 — Imports & seed
import torch, numpy as np, os, random
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn as nn
import wandb

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('CUDA available:', torch.cuda.is_available())

Device: cuda
CUDA available: True


In [5]:
# Cell 4 — Config
class Config:
    # paths — set after download
    train_img_dir   = ''
    train_mask_dir  = ''
    val_img_dir     = ''
    val_mask_dir    = ''

    # model
    encoder         = 'efficientnet-b4'
    encoder_weights = 'imagenet'
    num_classes     = 1
    dropout         = 0.3

    # training
    epochs          = 15
    batch_size      = 8
    lr              = 5e-5
    img_size        = 512

    # preprocessing
    bg_keep_ratio   = 0.20

    # loss
    dice_weight     = 0.7
    bce_weight      = 0.3

    # misc
    seed            = 42
    checkpoint_path = '/content/best_model.pth'
    results_path    = '/content/results.json'

cfg = Config()
print(f'Encoder: {cfg.encoder} | LR: {cfg.lr} | Epochs: {cfg.epochs} | Batch: {cfg.batch_size}')

Encoder: efficientnet-b4 | LR: 5e-05 | Epochs: 15 | Batch: 8


In [6]:
# Cell 5 — Inspect dataset structure & set paths
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:3]:
            print(f'  {indent}{f}')

1/
  wheat_rust_patches/
    processing_stats.json
    README.md
    val/
      masks/
      images/
    train/
      masks/
      images/
    test/
      masks/
      images/


In [7]:
cfg.train_img_dir  = os.path.join(path, 'wheat_rust_patches', 'train', 'images')
cfg.train_mask_dir = os.path.join(path, 'wheat_rust_patches', 'train', 'masks')
cfg.val_img_dir    = os.path.join(path, 'wheat_rust_patches', 'val', 'images')
cfg.val_mask_dir   = os.path.join(path, 'wheat_rust_patches', 'val', 'masks')

for p in [cfg.train_img_dir, cfg.train_mask_dir, cfg.val_img_dir, cfg.val_mask_dir]:
    print(p, '->', os.path.exists(p))

/root/.cache/kagglehub/datasets/abdur548/nwrd-patched/versions/1/wheat_rust_patches/train/images -> True
/root/.cache/kagglehub/datasets/abdur548/nwrd-patched/versions/1/wheat_rust_patches/train/masks -> True
/root/.cache/kagglehub/datasets/abdur548/nwrd-patched/versions/1/wheat_rust_patches/val/images -> True
/root/.cache/kagglehub/datasets/abdur548/nwrd-patched/versions/1/wheat_rust_patches/val/masks -> True


In [8]:
# Cell 6 — Background filtering & Dataset class
import cv2

def analyze_mask(mask_path):
    mask = np.array(Image.open(mask_path).convert('L'))
    mask = (mask > 127).astype(np.float32)
    return mask.sum() / mask.size

def filter_samples(img_dir, mask_dir, bg_keep_ratio=0.5):
    files = sorted(os.listdir(img_dir))
    disease, background = [], []
    for f in files:
        ratio = analyze_mask(os.path.join(mask_dir, f))
        (disease if ratio > 0 else background).append(f)
    kept_bg = random.sample(background, int(len(background) * bg_keep_ratio))
    print(f'Disease: {len(disease)} | BG kept: {len(kept_bg)}/{len(background)}')
    return disease + kept_bg

class WheatRustDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, mask_dir, files, transform=None):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.files     = files
        self.transform = transform

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        f    = self.files[idx]
        img  = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, f)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(os.path.join(self.mask_dir, f), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.unsqueeze(0)

In [9]:
# Cell 7 — Transforms & DataLoaders
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.ElasticTransform(p=0.5),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

train_files = filter_samples(cfg.train_img_dir, cfg.train_mask_dir, cfg.bg_keep_ratio)
val_files   = sorted(os.listdir(cfg.val_img_dir))

train_ds = WheatRustDataset(cfg.train_img_dir, cfg.train_mask_dir, train_files, train_tf)
val_ds   = WheatRustDataset(cfg.val_img_dir,   cfg.val_mask_dir,   val_files,   val_tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
imgs, masks = next(iter(train_loader))
print(f'Batch shape — images: {imgs.shape} | masks: {masks.shape}')

Disease: 2597 | BG kept: 1112/5563
Train batches: 464 | Val batches: 60
Batch shape — images: torch.Size([8, 3, 512, 512]) | masks: torch.Size([8, 1, 512, 512])


In [10]:
# Cell 8 — Loss function
def compute_pos_weight(loader):
    pos, neg = 0, 0
    for _, masks in loader:
        pos += masks.sum().item()
        neg += (1 - masks).sum().item()
    pw = neg / (pos + 1e-8)
    print(f'pos_weight: {pw:.2f}  (neg={neg:.0f}, pos={pos:.0f})')
    return torch.tensor([pw], device=device)

pos_weight = compute_pos_weight(train_loader)

class CombinedLoss(nn.Module):
    def __init__(self, dice_w=0.6, bce_w=0.4, pos_weight=None):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w  = bce_w
        self.dice   = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.bce    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, pred, target):
        return self.dice_w * self.dice(pred, target) + self.bce_w * self.bce(pred, target)

criterion = CombinedLoss(cfg.dice_weight, cfg.bce_weight, pos_weight)

pos_weight: 3.65  (neg=763386670, pos=208905426)


In [11]:
# Cell 12 — update freeze block to match CANet
for param in model.encoder.parameters():
    param.requires_grad = False
print("Encoder frozen for first 3 epochs")

for epoch in range(cfg.epochs):
    if epoch == 3:
        for param in model.encoder.parameters():
            param.requires_grad = True
        print("Encoder unfrozen")
    ...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

/tmp/ipykernel_1581/3422587618.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Output shape: torch.Size([2, 1, 512, 512])


In [12]:
# Cell 10 — Metrics
def compute_metrics(pred_logits, target, threshold=0.5):
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    tp = (pred * target).sum().item()
    fp = (pred * (1 - target)).sum().item()
    fn = ((1 - pred) * target).sum().item()
    tn = ((1 - pred) * (1 - target)).sum().item()
    return {
        'iou':         tp / (tp + fp + fn + 1e-8),
        'dice':        2*tp / (2*tp + fp + fn + 1e-8),
        'precision':   tp / (tp + fp + 1e-8),
        'recall':      tp / (tp + fn + 1e-8),
        'specificity': tn / (tn + fp + 1e-8),
        'accuracy':    (tp + tn) / (tp + tn + fp + fn + 1e-8),
    }

In [13]:
# Cell 11 — Train & validate epoch functions
def train_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss  = 0
    all_metrics = {k: 0 for k in ['iou','dice','precision','recall','specificity','accuracy']}
    for imgs, masks in tqdm(loader, desc='Train', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss  = criterion(preds, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        m = compute_metrics(preds.detach(), masks)
        for k in all_metrics: all_metrics[k] += m[k]
    n = len(loader)
    return total_loss / n, {k: v / n for k, v in all_metrics.items()}


def val_epoch(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss  = 0
    all_metrics = {k: 0 for k in ['iou','dice','precision','recall','specificity','accuracy']}
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='Val', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            with torch.amp.autocast('cuda'):
                preds = model(imgs)
                loss  = criterion(preds, masks)
            total_loss += loss.item()
            m = compute_metrics(preds, masks, threshold)
            for k in all_metrics: all_metrics[k] += m[k]
    n = len(loader)
    return total_loss / n, {k: v / n for k, v in all_metrics.items()}

In [14]:
# Cell 12 — W&B init + training loop
import wandb
os.environ['WANDB_API_KEY'] = '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'
wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)

wandb.init(project='wheat-rust-detection', config={
    'encoder':       cfg.encoder,
    'lr':            cfg.lr,
    'epochs':        cfg.epochs,
    'batch_size':    cfg.batch_size,
    'loss':          f'{cfg.dice_weight}*Dice + {cfg.bce_weight}*BCE',
    'bg_keep_ratio': cfg.bg_keep_ratio
})

history  = {k: [] for k in ['train_loss','val_loss','val_iou','val_dice']}
best_iou = 0.0

# Freeze encoder
for param in model.base.encoder.parameters():
    param.requires_grad = False
print("Encoder frozen for first 3 epochs")

for epoch in range(cfg.epochs):
    # Unfreeze at epoch 3
    if epoch == 3:
        for param in model.base.encoder.parameters():
            param.requires_grad = True
        print("Encoder unfrozen")

    tr_loss, tr_m = train_epoch(model, train_loader, optimizer, scaler, criterion)
    vl_loss, vl_m = val_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_iou'].append(vl_m['iou'])
    history['val_dice'].append(vl_m['dice'])

    wandb.log({
        'epoch':           epoch + 1,
        'train_loss':      tr_loss,
        'val_loss':        vl_loss,
        'val_iou':         vl_m['iou'],
        'val_dice':        vl_m['dice'],
        'val_precision':   vl_m['precision'],
        'val_recall':      vl_m['recall'],
        'val_specificity': vl_m['specificity'],
        'val_accuracy':    vl_m['accuracy'],
        'lr':              optimizer.param_groups[0]['lr']
    })

    print(f"Epoch {epoch+1:02d}/{cfg.epochs} | "
          f"TrLoss: {tr_loss:.4f} | VlLoss: {vl_loss:.4f} | "
          f"IoU: {vl_m['iou']:.4f} | Dice: {vl_m['dice']:.4f}")

    if vl_m['iou'] > best_iou:
        best_iou = vl_m['iou']
        torch.save({
            'epoch':     epoch + 1,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'iou':       best_iou
        }, cfg.checkpoint_path)
        print(f'  ✓ Saved best model (IoU: {best_iou:.4f})')

wandb.finish()
print(f'\nDone. Best Val IoU: {best_iou:.4f}')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: syedabdurrehman548 (syedabdurrehman548-nust) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Encoder frozen for first 3 epochs


Epoch 01/15 | TrLoss: 0.5967 | VlLoss: 0.3878 | IoU: 0.3915 | Dice: 0.4845
  ✓ Saved best model (IoU: 0.3915)


Epoch 02/15 | TrLoss: 0.5303 | VlLoss: 0.3525 | IoU: 0.4126 | Dice: 0.5012
  ✓ Saved best model (IoU: 0.4126)


Epoch 03/15 | TrLoss: 0.5267 | VlLoss: 0.3388 | IoU: 0.4136 | Dice: 0.5017
  ✓ Saved best model (IoU: 0.4136)
Encoder unfrozen


...

Epoch 13/15 | TrLoss: 0.3850 | VlLoss: 0.1938 | IoU: 0.5499 | Dice: 0.6198


Epoch 14/15 | TrLoss: 0.3671 | VlLoss: 0.1954 | IoU: 0.5490 | Dice: 0.6181


Epoch 15/15 | TrLoss: 0.3746 | VlLoss: 0.1958 | IoU: 0.5467 | Dice: 0.6168


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▇▆▃▂██▇▇▆▄▃▂▂▁█
train_loss,█▆▆▅▅▄▃▃▂▂▂▁▂▁▁
val_accuracy,▁▂▂▃▅▇▇█▇██████
val_dice,▁▂▂▃▄▆▆████▇███
val_iou,▁▂▂▃▄▆▆████▇███
val_loss,█▇▆▅▅▃▂▂▂▁▁▂▁▁▁
val_precision,▁▂▂▃▄▆▆█▇▇███▇▇
val_recall,▂▁▂▃▃▇▇▆█▆█▅█▇▇
val_specificity,▁▃▂▃▅▇▆█▇███▇██
epoch,15



Done. Best Val IoU: 0.5564




---



In [19]:
# Cell 13 — Threshold optimisation
import pandas as pd

ckpt = torch.load(cfg.checkpoint_path)
model.load_state_dict(ckpt['model'])
model.eval()

all_preds, all_masks = [], []
with torch.no_grad():
    for imgs, masks in val_loader:
        preds = torch.sigmoid(model(imgs.to(device))).cpu()
        all_preds.append(preds)
        all_masks.append(masks)
all_preds = torch.cat(all_preds)
all_masks = torch.cat(all_masks)

rows = []
for t in np.arange(0.3, 0.75, 0.05):
    pb = (all_preds > t).float()
    tp = (pb * all_masks).sum().item()
    fp = (pb * (1 - all_masks)).sum().item()
    fn = ((1 - pb) * all_masks).sum().item()
    rows.append({'threshold': round(t, 2), 'iou': round(tp / (tp + fp + fn + 1e-8), 4)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
best_thresh = df.loc[df['iou'].idxmax(), 'threshold']
print(f'\nBest threshold: {best_thresh}')

 threshold    iou
      0.30 0.6981
      0.35 0.7108
      0.40 0.7221
      0.45 0.7324
      0.50 0.7409
      0.55 0.7480
      0.60 0.7543
      0.65 0.7593
      0.70 0.7624

Best threshold: 0.7


In [20]:
# Cell 14 — Final evaluation

def tta_predict(model, img_tensor, device):
    transforms_list = [
        lambda x: x,
        lambda x: torch.flip(x, [-1]),
        lambda x: torch.flip(x, [-2]),
        lambda x: torch.rot90(x, 1, [-2,-1]),
    ]
    preds = []
    with torch.no_grad():
        for tf in transforms_list:
            inp = tf(img_tensor).unsqueeze(0).to(device)
            out = torch.sigmoid(model(inp)).squeeze(0).cpu()
            preds.append(tf(out))
    return torch.stack(preds).mean(0)


all_preds, all_masks = [], []
with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='TTA inference'):
    for i in range(imgs.shape[0]):
        pred = tta_predict(model, imgs[i], device)
        all_preds.append(pred)
    all_masks.append(masks)

all_preds = torch.stack(all_preds).unsqueeze(1)  # (N,1,512,512)
all_masks  = torch.cat(all_masks)

pb = (all_preds > best_thresh).float()
tp = (pb * all_masks).sum().item()
fp = (pb * (1 - all_masks)).sum().item()
fn = ((1 - pb) * all_masks).sum().item()
tn = ((1 - pb) * (1 - all_masks)).sum().item()

final = {
    'IoU':         round(tp / (tp + fp + fn + 1e-8), 4),
    'Dice/F1':     round(2*tp / (2*tp + fp + fn + 1e-8), 4),
    'Precision':   round(tp / (tp + fp + 1e-8), 4),
    'Recall':      round(tp / (tp + fn + 1e-8), 4),
    'Specificity': round(tn / (tn + fp + 1e-8), 4),
    'Accuracy':    round((tp + tn) / (tp + tn + fp + fn + 1e-8), 4),
}

targets = {'IoU': 0.83, 'Dice/F1': 0.90, 'Accuracy': 0.90, 'Specificity': 0.95}
print('\n=== Final Metrics ===')
for k, v in final.items():
    target = targets.get(k)
    status = f"-> {'PASS' if target and v >= target else 'FAIL' if target else ''}"
    print(f'  {k:12s}: {v:.4f}  {status}')

IndentationError: expected an indented block after 'for' statement on line 21 (3533571701.py, line 22)

In [ ]:
# Cell 15 — Plots & visualisation
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['val_iou'],  label='IoU')
ax2.plot(history['val_dice'], label='Dice')
ax2.set_title('Val Metrics'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

# Confusion matrix
cm = confusion_matrix(
    all_masks.numpy().flatten().astype(int),
    pb.numpy().flatten().astype(int)
)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred BG', 'Pred Disease'],
            yticklabels=['True BG', 'True Disease'])
plt.title('Confusion Matrix'); plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

# Qualitative 4-panel
def denorm(t):
    m, s = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    t = t.clone()
    for i in range(3): t[i] = t[i] * s[i] + m[i]
    return t.clamp(0, 1)

sample_imgs, sample_masks = next(iter(val_loader))
with torch.no_grad():
    sample_preds = torch.sigmoid(model(sample_imgs[:8].to(device))).cpu()

fig, axes = plt.subplots(8, 4, figsize=(16, 32))
for ax, col in zip(axes[0], ['Original', 'Ground Truth', 'Prob Map', 'Binary Mask']):
    ax.set_title(col, fontsize=11)
for i in range(8):
    axes[i][0].imshow(denorm(sample_imgs[i]).permute(1, 2, 0).numpy())
    axes[i][1].imshow(sample_masks[i][0].numpy(), cmap='gray')
    axes[i][2].imshow(sample_preds[i][0].numpy(), cmap='hot')
    axes[i][3].imshow((sample_preds[i][0] > best_thresh).numpy(), cmap='gray')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/qualitative_results.png', dpi=150)
plt.show()

In [ ]:
# Cell 16 — Save & download results
import json
from google.colab import files

results = {
    'best_val_iou':   best_iou,
    'best_threshold': float(best_thresh),
    'final_metrics':  final
}
with open(cfg.results_path, 'w') as f:
    json.dump(results, f, indent=2)

files.download(cfg.results_path)
files.download('/content/training_curves.png')
files.download('/content/confusion_matrix.png')
files.download('/content/qualitative_results.png')
files.download(cfg.checkpoint_path)